In [1]:
import torch
import torch.nn.functional as F

Muon (MomentUm Orthogonalized by Newton-Schulz) optimizes 2D neural network parameters by taking the updates generated by SGD-momentum, and then applying a Newton-Schulz (NS) iteration as a post-processing step to each of them before applying them to the parameters.

- orthogonal matrix: $Q^TQ=I$
- 矩阵的 0 次幂（zeroth power）
    - 奇异值分解（SVD）中的正交部分
- zeropower_via_newtonschulz5
    - zeropower: 计算矩阵的零次幂。
    - via_newtonschulz: 通过“牛顿-舒尔茨（Newton-Schulz）”迭代算法来实现。这是一种避免直接进行昂贵的SVD分解的数值方法。
    - 5: 指的是迭代中使用了一个五次（quintic）多项式。

## Gradient Whitening

- 在标准的梯度下降中，我们沿着负梯度方向更新权重。但如果梯度的各个维度之间相关性很高，或者尺度差异很大，优化过程就会很慢。
- “白化”变换旨在解耦 (decorrelate) 梯度的各个维度，并将其尺度归一化 (normalize scale)，使得优化路径更直接、高效。

### SVD

任何一个矩阵 G 都可以进行奇异值分解（SVD）：
$$
G = U S V^T
$$
- U 和 V 是正交矩阵（Orthogonal Matrices）。它们的列向量是标准正交的。
- S 是一个对角矩阵，对角线上的值是奇异值（Singular Values），表示了 G 在各个主方向上的“拉伸”或“缩放”程度。

矩阵 G 的“零次幂” G^0 在这里的定义是
$$
G^0 = U S^0 V^T
$$ 

其中 $S^0$ 是将 S 的所有非零对角元（奇异值）都替换为1得到的对角矩阵。因此，最终结果是：

$$
G^0 = U I V^T = UV^T
$$

这个 UV^T 矩阵是一个正交矩阵，它保留了原始矩阵 G 的“旋转”或“方向”信息，但完全丢弃了其“缩放”或“大小”的信息（因为所有奇异值都变成了1）。在信号处理和机器学习中，这个过程被称为白化（Whitening），因为它使得变换后的数据在各个方向上的方差都相等（均为1）。

### 牛顿-舒尔茨迭代

直接计算SVD（torch.linalg.svd）对于大矩阵来说计算成本很高。因此，作者采用了一种更快的迭代逼近方法：牛顿-舒尔茨迭代。



In [2]:
@torch.compile
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G.
    """
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps) # ensure top singular value <= 1
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.float() # 返回 float 方便对比

- 矩阵的 Frobenius Norm （F范数）总是大于或等于其谱范数（Spectral Norm，即最大奇异值）。
    - 对于矩阵 $X$ 奇异值假设为 $\sigma_1,\sigma_2, \cdots, \sigma_r$（降序排列 $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0$）
    - 谱范数：$||X||_2 = \sigma_{max} = \sigma_1$
        - $||X||_2^2 = \sigma_1^2$
    - F范数：$||X||_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n |x_{ij}|^2} = \sqrt{\sum_{k=1}^r \sigma_k^2}$
        - $||X||_F^2 = \sigma_1^2 + \sigma_2^2 + \dots + \sigma_r^2$
    - $||X||_F^2 \ge ||X||_2^2$
- `torch.norm`: defaut fro
    -  `X_norm = X / X.norm()`
        -  $||X_{norm}||_2 = ||\frac{1}{||X||_F} \cdot X||_2 = \frac{1}{||X||_F} \cdot ||X||_2 \leq \frac{1}{||X||_F} \cdot ||X||_F=1$
-  其他范数
    -  核范数（nuclear norm）
        - $||X||_* = \sum_{k=1}^r \sigma_k = \sigma_1 + \sigma_2 + \dots + \sigma_r$
        - 矩阵的迹 (Trace) 是其对角线元素之和，也等于其所有特征值 (Eigenvalues) 之和。
        - 矩阵的核范数 (Nuclear Norm) 是其所有奇异值 (Singular Values) 之和。
        - 对于一个半正定矩阵 $X^T X$，它的奇异值和特征值是相同的。核范数的严格定义是 $||X||_* = \text{Tr}(\sqrt{X^T X})$，即矩阵 $(X^T X)^{1/2}$ 的迹；

In [35]:
torch.manual_seed(123)
rows, cols = 4, 6
X = torch.randn(rows, cols) * 10 
X_f = X.norm()
X_f

tensor(46.1010)

In [36]:
sigmas = torch.linalg.svdvals(X)
sigmas

tensor([33.6294, 25.1017, 16.3428,  9.8583])

In [37]:
top_sigma = sigmas[0]

In [40]:
torch.sqrt(torch.sum(torch.linalg.svdvals(X) ** 2))

tensor(46.1010)

In [42]:
X_norm = X / X_f

In [43]:
torch.linalg.svdvals(X_norm)

tensor([0.7295, 0.5445, 0.3545, 0.2138])

In [44]:
# spectral_norm
torch.linalg.norm(X, ord=2)

tensor(33.6294)

In [46]:
torch.linalg.norm(X, ord='nuc')

tensor(84.9322)

In [48]:
torch.sum(torch.linalg.svdvals(X))

tensor(84.9322)

### 对比 SVD

In [3]:
def zeropower_via_svd(G):
    """
    Computes the zeroth power G^0 = UV^T using direct SVD.
    This is mathematically exact but computationally expensive.
    """
    # full_matrices=False 更高效，因为我们不需要完整的 U 或 V
    U, S, Vh = torch.linalg.svd(G, full_matrices=False)
    # G^0 = U @ Vh (因为 Vh 已经是 V.T)
    return U @ Vh

In [4]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
rows, cols = 256, 1024
G = torch.randn(rows, cols, device=device)

In [23]:
def diff(G, steps=3):
    exact_result = zeropower_via_svd(G)
    approx_result = zeropower_via_newtonschulz5(G, steps=steps)
    frobenius_diff = torch.norm(exact_result - approx_result)
    cosine_sim = F.cosine_similarity(exact_result.flatten(), approx_result.flatten(), dim=0)
    print(f'frobenius_diff: {frobenius_diff}, cos_sim: {cosine_sim}')
    s_exact = torch.linalg.svdvals(exact_result)
    s_approx = torch.linalg.svdvals(approx_result)
    print("\n   For the EXACT (SVD) result:")
    print(f"   - All singular values should be 1.0")
    print(f"   - Min: {s_exact.min().item():.4f}, Max: {s_exact.max().item():.4f}, Mean: {s_exact.mean().item():.4f}")
    print("\n   For the APPROXIMATE (Newton-Schulz) result:")
    print(f"   - Singular values should be scattered around 1.0")
    print(f"   - Min: {s_approx.min().item():.4f}, Max: {s_approx.max().item():.4f}, Mean: {s_approx.mean().item():.4f}")

In [25]:
diff(G, steps=3)

frobenius_diff: 2.364572763442993, cos_sim: 0.9932388067245483

   For the EXACT (SVD) result:
   - All singular values should be 1.0
   - Min: 1.0000, Max: 1.0001, Mean: 1.0001

   For the APPROXIMATE (Newton-Schulz) result:
   - Singular values should be scattered around 1.0
   - Min: 0.7570, Max: 1.2046, Mean: 1.0774


In [26]:
diff(G, steps=5)

frobenius_diff: 2.4109060764312744, cos_sim: 0.9889325499534607

   For the EXACT (SVD) result:
   - All singular values should be 1.0
   - Min: 1.0000, Max: 1.0001, Mean: 1.0001

   For the APPROXIMATE (Newton-Schulz) result:
   - Singular values should be scattered around 1.0
   - Min: 0.6846, Max: 1.1468, Mean: 0.9521


- orthogonal matrix$Q^TQ=I
$$